[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amaciasagro/GIT-RemoteSensing/blob/master/Climate/Climate_td_v2.ipynb)

# 🌦️ Climate Analyzer — v2
**Autor:** Ariel Macías | Agrónomo · GIS & Remote Sensing

Análisis climático sobre un lote agrícola usando **ERA5-Land** (ECMWF) via Google Earth Engine.

| Celda | Descripción |
|-------|-------------|
| 0 | Configuración (editar `config.py`) |
| 1 | Inicializar GEE y dibujar lote |
| 2 | Descarga ERA5 + métricas agronómicas + gráficos |

### Variables extraídas de ERA5-Land
| Variable | Descripción |
|----------|-------------|
| Precipitación | Suma diaria (mm) |
| T. máx / mín / media | Temperatura a 2 m (°C) |
| Punto de rocío | → Humedad Relativa estimada (%) |
| Radiación solar | Descendente superficial (MJ/m²/día) |
| Viento (u, v) | Componentes a 10 m → velocidad escalar (m/s) |
| ET vegetación | Evapotranspiración real ERA5 (mm) |
| **ETo Penman-Monteith** | Calculada con FAO-56 a partir de las anteriores |
| **Balance hídrico** | Lluvia acum. − ETo acum. (mm) |
| **GDA** | Grados día acumulados sobre T_BASE (config.py) |

In [1]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# Editá estas variables antes de correr el notebook.
# Para mayor seguridad podés cargar el proyecto desde entorno:
#   import os; GEE_PROJECT = os.environ.get('GEE_PROJECT', 'tu-proyecto')
# ============================================================

GEE_PROJECT  = 'my-project-12126-484118'   # <-- Reemplazá con tu project ID de GEE
CENTRO_LAT   = 33.584              # Latitud del centro del mapa inicial
CENTRO_LON   = -101.845            # Longitud del centro del mapa inicial
ZOOM_INICIAL = 14

print("✅ Configuración cargada.")
print(f"   Proyecto GEE : {GEE_PROJECT}")
print(f"   Centro mapa  : ({CENTRO_LAT}, {CENTRO_LON})")

✅ Configuración cargada.
   Proyecto GEE : my-project-12126-484118
   Centro mapa  : (33.584, -101.845)


---
## Paso 1 · Inicializar GEE y dibujar el lote
Usá la herramienta de polígono (panel izquierdo) para delimitar el lote. Luego ejecutá la Celda 2.

In [3]:
# ============================================================
# CELDA 1 — INICIALIZACIÓN GEE Y DIBUJO / CARGA DEL LOTE
# ============================================================
import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from IPython.display import display, HTML, FileLink
from contextlib import redirect_stdout
import json
import tempfile
import os

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# --- Autenticación GEE ---
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")
except Exception:
    print("🔑 Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")

✅ Earth Engine inicializado.


In [ ]:
# ============================================================\n
# STEP 1: GEE AUTH & AOI DEFINITION
# ============================================================\n

# --- Shared Global Variables ---
field_geom_ee  = None   # ee.Geometry object
field_geom_shp = None   # shapely geometry object

# --- Interactive Map Setup ---
interactive_map = geemap.Map(center=[CENTRO_LAT, CENTRO_LON], zoom=ZOOM_INICIAL)
interactive_map.add_basemap('SATELLITE')

# --- UI Widgets: Drawing ---
btn_confirm_draw = widgets.Button(
    description='✅ Confirm Drawn Polygon',
    button_style='success',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_draw = widgets.Output()

def confirm_polygon(b):
    global field_geom_ee, field_geom_shp
    with out_draw:
        out_draw.clear_output()
        roi = interactive_map.user_roi
        if roi is None:
            print("⚠️ No polygon detected. Please draw one first.")
            return

        roi_info       = roi.getInfo()
        field_geom_ee  = ee.Geometry(roi_info.get('geometry', roi_info))
        coords         = field_geom_ee.coordinates().getInfo()[0]
        field_geom_shp = Polygon(coords)

        area_ha = field_geom_shp.area * 111320**2 / 10000
        print(f"✅ Polygon confirmed: {len(coords)-1} vertices")
        print(f"   Estimated Area: {area_ha:.2f} ha")

btn_confirm_draw.unobserve_all()
btn_confirm_draw.on_click(confirm_polygon)

btn_download_shp = widgets.Button(
    description='📥 Download Drawn Polygon (.shp)',
    button_style='primary',
    layout=widgets.Layout(width='260px', margin='4px'),
    disabled=False  # Disabled until polygon is confirmed
)
out_download = widgets.Output()

def download_shp(b):
    with out_download:
        out_download.clear_output()
        gdf = gpd.GeoDataFrame(
            [{'name': 'field'}],
            geometry=[field_geom_shp],
            crs='epsg:4326'
        )
        tmp_dir = tempfile.mkdtemp()
        shp_path = os.path.join(tmp_dir, 'field.shp')
        gdf.to_file(shp_path)
        
        # Zip all shapefile components
        zip_path = 'field_boundary.zip'
        with zipfile.ZipFile(zip_path, 'w') as z:
            for ext in ['shp', 'shx', 'dbf', 'prj']:
                fp = os.path.join(tmp_dir, f'field.{ext}')
                if os.path.exists(fp):
                    z.write(fp, f'field.{ext}')
        
        display(FileLink(zip_path, result_html_prefix='👉 '))

btn_download_shp.unobserve_all()
btn_download_shp.on_click(download_shp)

# --- UI Widgets: File Upload ---
uploader = widgets.FileUpload(
    description='📂 Upload Field (.shp / .geojson)',
    accept='.shp,.zip,.geojson',
    multiple=True,
    layout=widgets.Layout(width='260px', margin='4px')
)
btn_upload = widgets.Button(
    description='📌 Load File to Map',
    button_style='info',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_upload = widgets.Output()

def load_spatial_file(b):
    global field_geom_ee, field_geom_shp
    with out_upload:
        out_upload.clear_output()
        files = uploader.value

        if not files:
            print("⚠️ No file uploaded.")
            return

        tmp_dir = tempfile.mkdtemp()
        
        for file in files:
            name, content = file['name'], file['content']
            with open(os.path.join(tmp_dir, name), 'wb') as f:
                f.write(content)

        try:
            # File handling logic (extract zip if needed)
            zip_files = [f for f in os.listdir(tmp_dir) if f.endswith('.zip')]
            if zip_files:
                import zipfile
                with zipfile.ZipFile(os.path.join(tmp_dir, zip_files[0]), 'r') as z:
                    z.extractall(tmp_dir)
            
            valid_files = [f for f in os.listdir(tmp_dir) if f.endswith(('.shp', '.geojson'))]
            if not valid_files:
                print("❌ No valid spatial file found.")
                return

            gdf = gpd.read_file(os.path.join(tmp_dir, valid_files[0])).to_crs(epsg=4326)
            
            field_geom_shp = unary_union(gdf.geometry)
            field_geom_ee  = ee.Geometry(mapping(field_geom_shp))
            
            area_ha = field_geom_shp.area * 111320**2 / 10000
            area_ac = area_ha * 2.47105
            print(f"✅ File loaded. Estimated Area: {area_ha:.2f} ha, {area_ac:.2f} ac.")

            with redirect_stdout(io.StringIO()):
                interactive_map.add_gdf(gdf, layer_name="Uploaded Field", style={'color': '#f1c40f', 'fillOpacity': 0.2})
        except Exception as e:
            print(f"❌ Error processing file: {e}")

btn_upload.unobserve_all()
btn_upload.on_click(load_spatial_file)

btn_download_shp.disabled = False

# --- Render UI ---
display(widgets.HTML("<h3 style='margin:8px 0'>📍 AOI Definition Panel</h3>"))
display(interactive_map)
display(widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Option A: Draw</b>"), btn_confirm_draw, out_draw,btn_download_shp, out_download]),
    widgets.VBox([widgets.HTML("<b>Option B: Upload</b>"), uploader, btn_upload, out_upload])
]))

HTML(value="<h3 style='margin:8px 0'>📍 AOI Definition Panel</h3>")

Map(center=[33.584, -101.845], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

---
## Paso 2 · Descarga ERA5 · Métricas agronómicas · Gráficos
Descarga datos ERA5-Land para el lote dibujado, calcula ETo (Penman-Monteith FAO-56),
balance hídrico y grados día acumulados, y genera los paneles interactivos.

In [13]:
# ============================================================
# CELDA 2 — ANÁLISIS COMPLETO
# ============================================================
from shapely.geometry import shape
import pandas as pd

from gee_utils      import obtener_fecha_maxima, calcular_periodos, descargar_serie
from agro_metrics   import calcular_eto_pm, calcular_hr, agregar_mensual, procesar_diario
from plots          import mostrar_graficos
from config         import ERA5_COLLECTION, T_BASE

# 1. Verificar que se definió el lote
if field_geom_ee is None or field_geom_shp is None:
    raise ValueError("⚠️ No se definió ningún lote. Volvé al Step 1, dibujá o subí un archivo y confirmá.")

print("📐 Configurando geometría del lote...")

# 2. Geometría del lote (ya definida en Step 1)
lote_geom = field_geom_ee
centroid  = lote_geom.centroid(maxError=1).coordinates().getInfo()
latitud   = centroid[1]

# 3. Fechas automáticas
era5_base = ee.ImageCollection(ERA5_COLLECTION).filterBounds(lote_geom)
fecha_max = obtener_fecha_maxima(era5_base)
hist_start, hist_end, daily_start, daily_end = calcular_periodos(fecha_max)

print(f'📅 Histórico  : {hist_start} → {hist_end}')
print(f'📅 Mes actual : {daily_start} → {daily_end}')

# 4. Descarga ERA5 (area mean sobre el lote, no centroide)
df = descargar_serie(lote_geom, hist_start, hist_end)

# 5. Métricas agronómicas
df['eto'] = calcular_eto_pm(df, latitud)
df['hr']  = calcular_hr(df)

# 6. Agregación mensual
df_mensual = agregar_mensual(df)

# 7. Detalle diario del mes actual
df_diario = procesar_diario(df, daily_start, daily_end)
df_diario.attrs['t_base'] = T_BASE   # para label del gráfico

# 8. Métricas resumen (imprimir antes de los gráficos)
mes_actual = df_mensual.iloc[-1]
print(f'\n📊 Resumen del último mes disponible ({mes_actual["mes_año"].strftime("%b %Y")})')
print(f'   Lluvia      : {mes_actual["precip"]:.1f} mm')
print(f'   ETo PM      : {mes_actual["eto"]:.1f} mm')
print(f'   Balance     : {mes_actual["balance_hidro"]:+.1f} mm')
print(f'   T. media    : {mes_actual["t_med"]:.1f} °C')
print(f'   HR media    : {mes_actual["hr"]:.0f} %')

# 9. Gráficos
print('\n📈 Generando gráficos interactivos...')
mostrar_graficos(df_mensual, df_diario, daily_start, daily_end)

📐 Configurando geometría del lote...
📅 Histórico  : 2023-05-01 → 2026-05-04
📅 Mes actual : 2026-05-01 → 2026-05-04
📡 Descargando serie temporal ERA5-Land (20-40 seg aprox.)...
✅ 1100 días descargados (2023-05-01 → 2026-05-04)

📊 Resumen del último mes disponible (May 2026)
   Lluvia      : 0.6 mm
   ETo PM      : 20.0 mm
   Balance     : -19.4 mm
   T. media    : 16.4 °C
   HR media    : 46 %

📈 Generando gráficos interactivos...
